### Chunking in RAG — Why It Matters and How It Works

#### Why Do We Need Chunking?

LLMs have a **context window limit** — they can only process a fixed number of tokens at once. When your document is larger than that limit, you can't just feed the whole thing in. Chunking solves this by splitting large documents into smaller, manageable pieces.

Beyond the hard limit, there's also a quality reason: even if a model *could* take in 100 pages at once, retrieving only the **relevant small section** gives much more focused, accurate answers than flooding the model with irrelevant text.

#### The Typical RAG Flow

1. Split documents → chunks
2. Embed each chunk into vectors
3. Store in a vector DB
4. At query time, retrieve only the top-K most relevant chunks
5. Pass those chunks *(not the whole document)* to the LLM

---

##### What `chunk_size` and `chunk_overlap`?

##### `chunk_size`

The **maximum size of each chunk**, measured in characters (or sometimes tokens depending on the splitter).

```
Document: "The quick brown fox jumps over the lazy dog. It was a sunny day."

chunk_size = 30  →  ["The quick brown fox jumps ove", "r the lazy dog. It was a sunn", "y day."]
```

| Too Large | Too Small |
|---|---|
| May exceed context limits or carry too much noise | Chunks lose context and meaning, more retrieval calls needed |

---

#### `chunk_overlap`

The **number of characters shared between consecutive chunks**. It creates a sliding window so that sentences or ideas that fall on a boundary aren't lost.

```
chunk_size = 40, chunk_overlap = 10

Chunk 1: |<----------  40 chars  ---------->|
Chunk 2:                          |<-- 10 -->|<----------  40 chars  ---------->|
                                  ↑ overlap zone repeated in both chunks
```

Without overlap, a sentence split across two chunks would be **half-understood** by each. Overlap ensures continuity.

---

## Visual Example

Given the text:

> *"Machine learning is a subset of AI. AI enables computers to learn. Learning improves with more data."*

With `chunk_size=50` and `chunk_overlap=10`:

```
Chunk 1: "Machine learning is a subset of AI. AI enables"
Chunk 2: "AI enables computers to learn. Learning improves"
Chunk 3: "improves with more data."
          ↑ overlapping words carry context across boundaries
```

---

## Practical Guidelines

| Scenario | `chunk_size` | `chunk_overlap` |
|---|---|---|
| Dense technical docs | 512–1024 | 100–200 |
| General Q&A | 256–512 | 50–100 |
| Short FAQs / dialogues | 128–256 | 20–50 |
| Code files | 500–1500 | 0–100 |

> **Rule of thumb:** Set overlap to roughly **10–20% of your chunk size** — enough to preserve boundary context without redundant retrieval noise.

In [5]:
text = open(r"C:\Users\Ammar\Downloads\sample_doc.txt", "r").read()
print(text)

Machine learning is a branch of artificial intelligence focused on building systems that learn from data. These systems improve their performance over time without being explicitly programmed. There are many applications of machine learning, such as image classification, speech recognition, recommendation systems, and autonomous driving.

Supervised learning uses labeled data to train predictive models. It is commonly used for tasks like spam detection and sentiment analysis. Unsupervised learning, on the other hand, discovers hidden patterns in unlabeled data, such as customer clustering or anomaly detection.

Reinforcement learning involves agents making decisions by interacting with an environment. They receive rewards or penalties based on their actions and learn optimal behaviors through continuous feedback. This approach is widely used in robotics, game playing, and resource optimization.

Although machine learning is powerful, it also comes with challenges such as data bias, ove

## 1. Fixed-Size Chunking

In [22]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_text(text)

print("Fixed Size Chunks:", len(chunks))
for i, chunk in enumerate(chunks):
    print(f"Line {i}: {chunk}")

Created a chunk of size 339, which is longer than the specified 300


Fixed Size Chunks: 4
Line 0: Machine learning is a branch of artificial intelligence focused on building systems that learn from data. These systems improve their performance over time without being explicitly programmed. There are many applications of machine learning, such as image classification, speech recognition, recommendation systems, and autonomous driving.
Line 1: Supervised learning uses labeled data to train predictive models. It is commonly used for tasks like spam detection and sentiment analysis. Unsupervised learning, on the other hand, discovers hidden patterns in unlabeled data, such as customer clustering or anomaly detection.
Line 2: Reinforcement learning involves agents making decisions by interacting with an environment. They receive rewards or penalties based on their actions and learn optimal behaviors through continuous feedback. This approach is widely used in robotics, game playing, and resource optimization.
Line 3: Although machine learning is powerful, it

## 2. Recursive Character Chunking

In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)
chunks = splitter.split_text(text)

print("Recursive Chunks:", len(chunks))
for i, chunk in enumerate(chunks):
    print(f"Line {i}: {chunk}")

Recursive Chunks: 4
Line 0: Machine learning is a branch of artificial intelligence focused on building systems that learn from data. These systems improve their performance over time without being explicitly programmed. There are many applications of machine learning, such as image classification, speech recognition, recommendation systems, and autonomous driving.
Line 1: Supervised learning uses labeled data to train predictive models. It is commonly used for tasks like spam detection and sentiment analysis. Unsupervised learning, on the other hand, discovers hidden patterns in unlabeled data, such as customer clustering or anomaly detection.
Line 2: Reinforcement learning involves agents making decisions by interacting with an environment. They receive rewards or penalties based on their actions and learn optimal behaviors through continuous feedback. This approach is widely used in robotics, game playing, and resource optimization.
Line 3: Although machine learning is powerful, it 

## 3. Token-Based Chunking

In [36]:
import tiktoken
from langchain_text_splitters import TokenTextSplitter

splitter = TokenTextSplitter(chunk_size=256, chunk_overlap=32)
chunks = splitter.split_text(text)

print("Token-Based Chunks:", len(chunks))
print(chunks[0])

ImportError: Could not import tiktoken python package. This is needed in order to for TokenTextSplitter. Please install it with `pip install tiktoken`.